In [3]:
import os

dataset_path = '/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed'
images = os.listdir(dataset_path)
print(f"Total images: {len(images)}")
print("Sample:", images[:3])

Total images: 3023
Sample: ['Ether-5_png_jpg.rf.Bw5VPDuvZ2CoFbLZ4PVk.jpg', 'Antigen-1_png_jpg.rf.Gq7SALfR27kSf5hsZoeS.jpg', 'Probenicid-1_png_jpg.rf.oJO0YFJfLSx93J4SzHX6.jpg']


In [4]:
# install libraries
!pip install transformers jiwer -q

import os
import torch
import shutil
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.9 MB/s eta 0:00:00a 0:00:01
Using: cuda
GPU: Tesla T4


In [5]:
# test the label fix
test_files = [
    '-a-Benzyl-Penicillin-2_png_jpg.rf.xxx.jpg',
    'Ethosoximide-4_png_jpg.rf.xxx.jpg',
    'Benzyl-morphine-1_png_jpg.rf.xxx.jpg'
]

def get_actual_label(filename):
    name = filename.lstrip('-')
    parts = name.split('-')
    clean_parts = []
    for part in parts:
        if part.replace('_','').replace('png','').replace('jpg','').isdigit():
            break
        if 'png' in part.lower() or 'jpg' in part.lower():
            break
        clean_parts.append(part)
    return ' '.join(clean_parts).strip().lower()

for f in test_files:
    print(f"{f[:45]} → '{get_actual_label(f)}'")

-a-Benzyl-Penicillin-2_png_jpg.rf.xxx.jpg → 'a benzyl penicillin'
Ethosoximide-4_png_jpg.rf.xxx.jpg → 'ethosoximide'
Benzyl-morphine-1_png_jpg.rf.xxx.jpg → 'benzyl morphine'


In [6]:
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id

model = model.to(device)
print("TrOCR loaded!")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrOCR loaded!


In [13]:
from sklearn.model_selection import train_test_split
import os

# Recreate the split
image_folder = "/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/"
all_images = os.listdir(image_folder)

train_images, val_images = train_test_split(
    all_images,
    test_size=0.2,
    random_state=42   # ← keep this same number as before
)

print(f"Total   : {len(all_images)}")
print(f"Train   : {len(train_images)}")
print(f"Val     : {len(val_images)}")

Total   : 3023
Train   : 2418
Val     : 605


In [15]:
from sklearn.model_selection import train_test_split
import os

image_folder = "/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/"
all_images = os.listdir(image_folder)

# ✅ Fix 1 — Filter out empty labels before splitting
valid_images = [
    f for f in all_images 
    if get_actual_label(f).strip() != ''
]

print(f"Total images     : {len(all_images)}")
print(f"After filtering  : {len(valid_images)}")
print(f"Empty removed    : {len(all_images) - len(valid_images)}")

# ✅ Fix 2 — Make sure same split as training
train_images, val_images = train_test_split(
    valid_images,
    test_size=0.2,
    random_state=42  # must match what you used in training
)

print(f"Train            : {len(train_images)}")
print(f"Val              : {len(val_images)}")

Total images     : 3023
After filtering  : 2054
Empty removed    : 969
Train            : 1643
Val              : 411


In [16]:
# Check overlap between train and val
overlap = set(train_images) & set(val_images)
print(f"Overlap between train & val: {len(overlap)}")  
# Must be 0 — if not, your split is wrong

Overlap between train & val: 0


In [17]:
# ✅ Better filter — remove junk labels
def is_valid_label(label):
    label = label.strip()
    if len(label) <= 2:          # too short (single letters like "b")
        return False
    if any(char.isdigit() for char in label) and len(label) < 5:
        return False             # page numbers like "2_page"
    if 'page' in label.lower():  # explicitly remove page labels
        return False
    return True

valid_images = [
    f for f in all_images
    if is_valid_label(get_actual_label(f))
]

print(f"After cleaning: {len(valid_images)} valid images")

After cleaning: 1988 valid images


In [18]:
overlap = set(train_images) & set(val_images)
print(f"Train/Val overlap: {len(overlap)}")  # Must be 0

Train/Val overlap: 0


In [19]:
from sklearn.model_selection import train_test_split

train_images, val_images = train_test_split(
    valid_images,
    test_size=0.2,
    random_state=42
)

print(f"Total clean images : {len(valid_images)}")
print(f"Train              : {len(train_images)}")
print(f"Val                : {len(val_images)}")
# Expected: Train ~1590, Val ~398

Total clean images : 1988
Train              : 1590
Val                : 398


In [22]:
class PrescriptionDataset(Dataset):
    def __init__(self, image_list, processor, image_folder):
        # ✅ Accept a list of filenames + folder path separately
        self.images       = image_list
        self.processor    = processor
        self.image_folder = image_folder

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        filename      = self.images[idx]
        medicine_name = get_actual_label(filename).strip().lower()

        image_path = os.path.join(self.image_folder, filename)
        image      = Image.open(image_path).convert('RGB')

        pixel_values = self.processor(
            image, return_tensors='pt'
        ).pixel_values.squeeze()

        labels = self.processor.tokenizer(
            medicine_name,
            padding='max_length',
            max_length=64,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return pixel_values, labels
        

In [23]:
# Recreate datasets with clean images only
train_dataset = PrescriptionDataset(train_images, processor, image_folder)
val_dataset   = PrescriptionDataset(val_images,   processor, image_folder)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")

Train samples : 1590
Val samples   : 398


In [25]:
from transformers import VisionEncoderDecoderModel

# Load fresh model
model = VisionEncoderDecoderModel.from_pretrained(
    'microsoft/trocr-base-handwritten'
).to(device)

# ✅ Fix — configure token IDs
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# ✅ Fix — configure generation settings
model.config.max_length    = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams      = 4

print("Model configured successfully!")
print(f"  pad_token_id           : {model.config.pad_token_id}")
print(f"  decoder_start_token_id : {model.config.decoder_start_token_id}")
print(f"  eos_token_id           : {model.config.eos_token_id}")

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model configured successfully!
  pad_token_id           : 1
  decoder_start_token_id : 0
  eos_token_id           : 2


In [ ]:
from transformers import VisionEncoderDecoderModel, get_linear_schedule_with_warmup

# ── 1. Load model ──────────────────────────────────────────
model = VisionEncoderDecoderModel.from_pretrained(
    'microsoft/trocr-base-handwritten'
).to(device)

# ✅ Fix — use generation_config instead of model.config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# ← move these two here
model.generation_config.max_length  = 64
model.generation_config.num_beams   = 4

print(f"pad_token_id           : {model.config.pad_token_id}")
print(f"decoder_start_token_id : {model.config.decoder_start_token_id}")
print("Config OK — starting training...\n")

# ── 3. Optimizer & scheduler ────────────────────────────────
optimizer     = torch.optim.AdamW(model.parameters(), lr=2e-5)
total_steps   = len(train_dataloader) * 10
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = 100,
    num_training_steps = total_steps
)

# ── 4. Training loop ────────────────────────────────────────
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_idx, (pixel_values, labels) in enumerate(train_dataloader):
        pixel_values = pixel_values.to(device)
        labels       = labels.to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {avg_loss:.4f}")

print("\nTraining complete!")


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


pad_token_id           : 1
decoder_start_token_id : 0
Config OK — starting training...

Epoch 1/10 — Loss: 1.6484
Epoch 2/10 — Loss: 0.3441
Epoch 3/10 — Loss: 0.2304
Epoch 4/10 — Loss: 0.1534
Epoch 5/10 — Loss: 0.0817
Epoch 6/10 — Loss: 0.0519
Epoch 7/10 — Loss: 0.0313


In [ ]:
from difflib import SequenceMatcher
from sklearn.metrics import f1_score

model.eval()
results = []

print("Evaluating on clean val set...\n")

with torch.no_grad():
    for img_file in val_images:
        actual = get_actual_label(img_file).strip().lower()

        image = Image.open(
            os.path.join(image_folder, img_file)
        ).convert('RGB')

        pixel_values = processor(
            image, return_tensors='pt'
        ).pixel_values.to(device)

        generated_ids = model.generate(
            pixel_values,
            max_length = 64,
            num_beams  = 4
        )
        predicted = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip().lower()

        similarity = SequenceMatcher(None, actual, predicted).ratio()

        results.append({
            'actual'     : actual,
            'predicted'  : predicted,
            'similarity' : similarity,
            'exact_match': actual == predicted
        })

# Results
total    = len(results)
exact    = sum(r['exact_match'] for r in results)
avg_sim  = sum(r['similarity']  for r in results) / total
threshold = 0.80
correct  = [1 if r['similarity'] >= threshold else 0 for r in results]
f1       = f1_score([1]*total, correct, average='binary')

print("=" * 40)
print("     FINAL EVALUATION RESULTS")
print("=" * 40)
print(f"  Total Samples   : {total}")
print(f"  Exact Accuracy  : {exact/total*100:.1f}%")
print(f"  Avg Similarity  : {avg_sim*100:.1f}%")
print(f"  F1 Score (>80%) : {f1*100:.1f}%")
print("=" * 40)

print("\nSample Predictions:")
print(f"{'ACTUAL':<25} {'PREDICTED':<25} {'SCORE'}")
print("-" * 60)
for r in results[:15]:
    icon = "✅" if r['similarity'] >= threshold else "❌"
    print(f"{icon} {r['actual']:<23} {r['predicted']:<23} {r['similarity']*100:.0f}%")